In [38]:
import pandas as pd

Чтение csv

In [39]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

In [40]:
df=pd.read_csv(url)

первичный анализ

In [41]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

Смотрим первые пять строк

In [42]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Смотрим информацию о типах данных

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


Информация о численных переменных

In [44]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


анализ целевой переменной

In [45]:
df.groupby('gender').count()['customerID']

,customerID
gender,
Female,3488
Male,3555


In [46]:
df.groupby('SeniorCitizen').count()['customerID']

,customerID
SeniorCitizen,
0,5901
1,1142


In [47]:
mask=df['InternetService']=='Fiber optic'

In [48]:
df[mask].count()

,0
customerID,3096
gender,3096
SeniorCitizen,3096
Partner,3096
Dependents,3096
tenure,3096
PhoneService,3096
MultipleLines,3096
InternetService,3096
OnlineSecurity,3096


In [49]:
df['tenure'].mean()

np.float64(32.37114865824223)

In [50]:
mask=df['Churn']=='Yes'

In [54]:
need=df[mask]['SeniorCitizen'].sum()

In [55]:
all=len(df[mask])

In [57]:
need/all*100

np.float64(25.468164794007492)

In [59]:
df.groupby('Churn')['MonthlyCharges'].mean()

,MonthlyCharges
Churn,
No,61.265124
Yes,74.441332


In [78]:
PM=df.groupby("PaymentMethod")['customerID'].count().idxmax()

In [79]:
PM

'Electronic check'

In [68]:
PM[max(PM['customerID'])]

KeyError: 'customerID'

In [84]:
df_tenure=df[df['tenure']==0]
df_tenure.groupby('tenure')['customerID'].count()

,customerID
tenure,
0,11


In [87]:
len(df['customerID'])

7043

In [94]:
df_contract=df[df['Contract']=="Two year"]

In [98]:
ch_y=df_contract[df_contract['Churn']=='Yes']

In [99]:
len(ch_y)/len(df['customerID'])*100

0.6815277580576459

In [100]:
ch_y = df[(df['Contract'] == 'Two year') & (df['Churn'] == 'Yes')]
total_y = df[df['Contract'] == 'Two year']
len(ch_y) / len(total_y) * 100

2.831858407079646

In [102]:
df.isna().sum()

,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0
OnlineBackup,0


In [101]:
df = df.drop('customerID', axis=1)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

In [121]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, classification_report

X = df.drop('Churn', axis=1)

In [104]:
y=df['Churn']

In [105]:
X=pd.get_dummies(X,drop_first=True,dtype=int)

In [108]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [114]:
scaler = StandardScaler()

In [115]:
X_train_scaled = scaler.fit_transform(X_train)

In [116]:
X_test_scaled = scaler.transform(X_test)

In [117]:
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

LogisticRegression()

In [118]:
y_pred = model.predict(X_test_scaled)

In [119]:
y_proba = model.predict_proba(X_test_scaled)[:, 1]

In [123]:
y_pred_custom = (y_proba > 0.3).astype(int)

In [122]:
print("=== МЕТРИКИ МОДЕЛИ ===")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_proba):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred):.4f}")
print("\n=== ПОДРОБНЫЙ ОТЧЕТ ===")
print(classification_report(y_test, y_pred))

=== МЕТРИКИ МОДЕЛИ ===
ROC-AUC:  0.8621
F1-score: 0.6370

=== ПОДРОБНЫЙ ОТЧЕТ ===
              precision    recall  f1-score   support

           0       0.86      0.90      0.88      1036
           1       0.69      0.60      0.64       373

    accuracy                           0.82      1409
   macro avg       0.77      0.75      0.76      1409
weighted avg       0.81      0.82      0.82      1409



In [137]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,          # количество деревьев
    max_depth=10,              # ограничение глубины (защита от переобучения)
    class_weight='balanced',   # ВАЖНО! Учитываем дисбаланс классов
    random_state=42
)
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [138]:
y_pred_ensemble=model.predict(X_test)
y_proba_ensemble = model.predict_proba(X_test)[:, 1]

In [139]:
print("Train features:", X_train.shape[1])
print("Test features:", X_test.shape[1])

Train features: 30
Test features: 30


In [140]:
print(f"ROC-AUC:  {roc_auc_score(y_test, y_proba_ensemble):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_ensemble):.4f}")


ROC-AUC:  0.8372
F1-score: 0.5367
